In [11]:
import mlflow
import numpy as np
import plotly.graph_objects as go
import torch
import torch.nn as nn
import torch.optim as optim

device = "cuda" if torch.cuda.is_available() else "cpu"
mlflow.set_tracking_uri("../mlruns")
mlflow.set_experiment("pinn")

<Experiment: artifact_location='/home/jean/projects/studies/notebooks/../mlruns/844625744777213672', creation_time=1765206373343, experiment_id='844625744777213672', last_update_time=1765206373343, lifecycle_stage='active', name='pinn', tags={'mlflow.experimentKind': 'custom_model_development'}>

Heat equation 2d

$$u_t = u_{xx} + u_{yy}$$
$$u(x,y,0) = sin(\pi x)sin(\pi y)$$
$$u(-1,y,t) = 0$$
$$u(1,y,t) = 0$$
$$u(x,-1,t) = 0$$
$$u(x,1,t) = 0$$


In [12]:
x_f = np.random.uniform(-1, 1, 10000)
y_f = np.random.uniform(-1, 1, 10000)
t_f = np.random.uniform(0, 1, 10000)
x_i = np.random.uniform(-1, 1, 250)
y_i = np.random.uniform(-1, 1, 250)
t_i = np.zeros_like(x_i)
u_i = np.sin(np.pi * x_i) * np.sin(np.pi * y_i)
t_b = np.random.uniform(0, 1, 250)
x_bl = -np.ones_like(t_b)
y_bl = np.random.uniform(-1, 1, 250)
x_br = np.ones_like(t_b)
y_br = np.random.uniform(-1, 1, 250)
x_bu = np.random.uniform(-1, 1, 250)
x_bd = np.random.uniform(-1, 1, 250)
y_bu = -np.ones_like(x_bu)
y_bd = np.ones_like(x_bd)
u_b = np.zeros_like(t_b)

In [13]:
def calculate_grad(f: torch.Tensor, x: torch.Tensor, retain_graph: bool = True) -> torch.Tensor:
    return  torch.autograd.grad(
            f, x,
            grad_outputs=torch.ones_like(f),
            retain_graph=retain_graph,
            create_graph=True
        )[0]

In [14]:
class HeatBarPINN(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.input = nn.Sequential(
            nn.Linear(3, 50),
            nn.Tanh()
        )

        hidden_layers = []
        for _ in range(8):
            hidden_layers.append(nn.Linear(50, 50))
            hidden_layers.append(nn.Tanh())
        self.hidden = nn.Sequential(*hidden_layers)

        self.output = nn.Sequential(
            nn.Linear(50, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.input(x)
        x = self.hidden(x)
        x = self.output(x)
        return x
    
    def pde_residual(
            self,
            x_f: torch.Tensor,
            y_f: torch.Tensor,
            t_f: torch.Tensor
    ) -> torch.Tensor:
        u_f = self.forward(torch.cat([x_f, y_f, t_f], dim=1))
        u_t = calculate_grad(u_f, t_f)
        u_x = calculate_grad(u_f, x_f)
        u_y = calculate_grad(u_f, y_f)
        u_xx = calculate_grad(u_x, x_f)
        u_yy = calculate_grad(u_y, y_f)

        residual = u_t - u_xx - u_yy
        return residual
    

class MLflowCallback:
    def __init__(self) -> None:
        pass
    
    def on_epoch_end(self, epoch: int, losses_dict: dict[str, float]) -> None:
        for name, value in losses_dict.items():
            mlflow.log_metric(name, value, step=epoch)

    def log_params(self, params_dict: dict[str, int | float]) -> None:
        for name, value in params_dict.items():
            mlflow.log_param(name, value)

In [15]:
model = HeatBarPINN().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=100, 
    min_lr=1e-6
)
epochs = 2000

In [16]:
x_f = torch.tensor(x_f, dtype=torch.float32, requires_grad=True).reshape(-1, 1).to(device)
y_f = torch.tensor(y_f, dtype=torch.float32, requires_grad=True).reshape(-1, 1).to(device)
t_f = torch.tensor(t_f, dtype=torch.float32, requires_grad=True).reshape(-1, 1).to(device)
x_i = torch.tensor(x_i, dtype=torch.float32).reshape(-1, 1).to(device)
y_i = torch.tensor(y_i, dtype=torch.float32).reshape(-1, 1).to(device)
t_i = torch.tensor(t_i, dtype=torch.float32).reshape(-1, 1).to(device)
u_i = torch.tensor(u_i, dtype=torch.float32).reshape(-1, 1).to(device)
t_b = torch.tensor(t_b, dtype=torch.float32).reshape(-1, 1).to(device)
x_bl = torch.tensor(x_bl, dtype=torch.float32).reshape(-1, 1).to(device)
y_bl = torch.tensor(y_bl, dtype=torch.float32).reshape(-1, 1).to(device)
x_br = torch.tensor(x_br, dtype=torch.float32).reshape(-1, 1).to(device)
y_br = torch.tensor(y_br, dtype=torch.float32).reshape(-1, 1).to(device)
x_bu = torch.tensor(x_bu, dtype=torch.float32).reshape(-1, 1).to(device)
x_bd = torch.tensor(x_bd, dtype=torch.float32).reshape(-1, 1).to(device)
y_bu = torch.tensor(y_bu, dtype=torch.float32).reshape(-1, 1).to(device)
y_bd = torch.tensor(y_bd, dtype=torch.float32).reshape(-1, 1).to(device)
u_b = torch.tensor(u_b, dtype=torch.float32).reshape(-1, 1).to(device)

In [ ]:
with mlflow.start_run():
    callback = MLflowCallback()
    params = {
        "lr": 1e-3,
        "epochs": epochs,
    }
    callback.log_params(params)
    for epoch in range(epochs):
        optimizer.zero_grad()
        pred_ui = model(torch.cat([x_i, y_i, t_i], dim=1))
        loss_ic = criterion(pred_ui, u_i)

        pred_ubl = model(torch.cat([x_bl, y_bl, t_b], dim=1))
        pred_ubr = model(torch.cat([x_br, y_br, t_b], dim=1))
        pred_ubu = model(torch.cat([x_bu, y_bu, t_b], dim=1))
        pred_ubd = model(torch.cat([x_bd, y_bd, t_b], dim=1))
        loss_bc = (
            criterion(pred_ubl, u_b)
            + criterion(pred_ubr, u_b)
            + criterion(pred_ubu, u_b)
            + criterion(pred_ubd, u_b)
        )

        residual = model.pde_residual(x_f, y_f, t_f)
        loss_pde = torch.mean(residual**2)

        loss = 10*loss_ic + 10*loss_bc + loss_pde
        losses = {
            "loss_ic": loss_ic.item(),
            "loss_bc": loss_bc.item(),
            "loss_pde": loss_pde.item(),
            "loss": loss.item(),
        }
        callback.on_epoch_end(epoch, losses)
        loss.backward()
        optimizer.step()
        
        scheduler.step(loss.item())


In [18]:
x = np.linspace(-1, 1, 50)
y = np.linspace(-1, 1, 50)
t = np.linspace(0, 1, 50)
x_mesh, y_mesh, t_mesh = np.meshgrid(x, y, t)
mesh = np.stack([x_mesh.flatten(), y_mesh.flatten(), t_mesh.flatten()], axis=1)
mesh = torch.tensor(mesh, dtype=torch.float32).to(device)

In [19]:
model.eval()
with torch.no_grad():
    u_pred = model(mesh).cpu().numpy().reshape(50, 50, 50)

Analytic solution
$$u(x,y,t) = e^{-2\pi^2t}sin(\pi x)sin(\pi y)$$

In [20]:
u_ana = np.exp(-2 * (np.pi ** 2) * t_mesh) * np.sin(np.pi * x_mesh) * np.sin(np.pi * y_mesh)
rmse = np.sqrt(np.mean((u_pred - u_ana)**2, axis=(0,1)))
rmse

array([0.04166145, 0.02458752, 0.01835152, 0.01609497, 0.01444164,
       0.01272571, 0.01101534, 0.00944745, 0.00810706, 0.00702792,
       0.00620757, 0.00561876, 0.00521873, 0.00495891, 0.00479376,
       0.0046865 , 0.00461055, 0.00454846, 0.00448966, 0.00442832,
       0.00436168, 0.00428887, 0.0042101 , 0.00412618, 0.0040382 ,
       0.00394737, 0.00385488, 0.00376189, 0.00366948, 0.00357861,
       0.00349017, 0.00340495, 0.00332367, 0.00324695, 0.00317533,
       0.0031093 , 0.00304923, 0.00299546, 0.0029482 , 0.00290762,
       0.00287379, 0.00284669, 0.00282624, 0.00281226, 0.00280453,
       0.00280274, 0.00280655, 0.00281557, 0.00282937, 0.00284752])